In [11]:
import timm
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model('vit_tiny_patch16_224', pretrained=True)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

model = model.to(device)
print(f"Using device: {device}")

Using device: cuda


In [12]:
import torch.nn as nn

num_classes = 2  # malware vs benign
model.head = nn.Linear(model.head.in_features, num_classes).to(device)
# head params are unfrozen by default since it's newly created

In [ ]:
path_to_dataset = "/home/dc_gr1/kaushik/vit_transformer/ExeImg_Dataset"

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import random

transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(path_to_dataset, transform=transform)

# Use 40% subset
subset_size = int(0.4 * len(full_dataset))
indices = random.sample(range(len(full_dataset)), subset_size)
dataset = Subset(full_dataset, indices)

print(f"Using {len(dataset)} / {len(full_dataset)} samples ({subset_size/len(full_dataset)*100:.0f}%)")

dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

Using 11786 / 29466 samples (40%)


In [14]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

model.train()

print("Device:", device)

for epoch in range(5):
    start_time = time.time()

    for i, (images, labels) in enumerate(dataloader):

        # 🔍 Debug: check batch loading
        if i == 0:
            print("First batch loaded")
            print("Shape:", images.shape)

        # 🚀 Move to GPU
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # 🔍 Debug: confirm GPU usage
        if i == 0:
            print("Device after transfer:", images.device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 🔍 Progress print (VERY IMPORTANT)
        if i % 50 == 0:
            print(f"Epoch {epoch}, Batch {i}, Loss {loss.item():.4f}")

    end_time = time.time()
    print(f"Epoch {epoch} done in {end_time - start_time:.2f} seconds")

Device: cuda
First batch loaded
Shape: torch.Size([64, 3, 224, 224])
Device after transfer: cuda:0
Epoch 0, Batch 0, Loss 1.6117
Epoch 0, Batch 50, Loss 0.6830
Epoch 0, Batch 100, Loss 0.4640
Epoch 0, Batch 150, Loss 0.3638
Epoch 0 done in 9.65 seconds
First batch loaded
Shape: torch.Size([64, 3, 224, 224])
Device after transfer: cuda:0
Epoch 1, Batch 0, Loss 0.4142
Epoch 1, Batch 50, Loss 0.2310
Epoch 1, Batch 100, Loss 0.2020
Epoch 1, Batch 150, Loss 0.3232
Epoch 1 done in 9.41 seconds
First batch loaded
Shape: torch.Size([64, 3, 224, 224])
Device after transfer: cuda:0
Epoch 2, Batch 0, Loss 0.2545
Epoch 2, Batch 50, Loss 0.2854
Epoch 2, Batch 100, Loss 0.1783
Epoch 2, Batch 150, Loss 0.2976
Epoch 2 done in 9.45 seconds
First batch loaded
Shape: torch.Size([64, 3, 224, 224])
Device after transfer: cuda:0
Epoch 3, Batch 0, Loss 0.1707
Epoch 3, Batch 50, Loss 0.1871
Epoch 3, Batch 100, Loss 0.2415
Epoch 3, Batch 150, Loss 0.1572
Epoch 3 done in 9.47 seconds
First batch loaded
Shape: t

In [15]:
torch.save(model.state_dict(), "vit_malware.pth")

In [16]:
block = model.blocks[0]  # pick one block

W1 = block.mlp.fc1.weight.detach().cpu()
b1 = block.mlp.fc1.bias.detach().cpu()

W2 = block.mlp.fc2.weight.detach().cpu()
b2 = block.mlp.fc2.bias.detach().cpu()

In [17]:
scale = 256  # 2^8

W1_fixed = (W1 * scale).round().int()
W2_fixed = (W2 * scale).round().int()
b1_fixed = (b1 * scale).round().int()
b2_fixed = (b2 * scale).round().int()

In [18]:
import numpy as np

np.savetxt("W1.mem", W1_fixed.numpy().flatten(), fmt="%d")
np.savetxt("W2.mem", W2_fixed.numpy().flatten(), fmt="%d")

In [19]:
# simple numpy GEMM check
import numpy as np

x = torch.randn(1, W1.shape[1])

y1 = x @ W1.T + b1
y2 = torch.nn.functional.gelu(y1)
y3 = y2 @ W2.T + b2